# Audit: what does `tabpfn` do by default?

**CPU only. No model weights downloaded. Runs in seconds.**

This notebook inspects the *installed* `tabpfn` package - signatures, default
values, docstrings, constants - and prints file paths and line numbers so that
every claim in the README can be checked against the source.

**What to look at.** Section 2 prints the defaults of `FinetunedTabPFNClassifier`.
The three that matter:

* `n_finetune_ctx_plus_query_samples` - a cap on context + query per episode
* `finetune_ctx_query_split_ratio` - the query share carved out of it
* `n_inference_subsample_samples` - `None` means the full training set is context

Together they lock the ratio at **1.25** for any training set below the cap, and
release it above. Section 5 classifies the finding automatically.

*The body of this notebook is in French, the language the research was conducted
in. The printed output is what matters and is self-explanatory.*

---


# Que fait `tabpfn` par défaut ? — taille du contexte au fine-tuning et à l'inférence**Aucun GPU requis, aucun poids téléchargé.** Ce notebook ne fait que de l'introspection :signatures, valeurs par défaut, docstrings et constantes du paquet installé. Il tourne enquelques secondes et n'instancie aucun modèle, donc il peut tourner pendant qu'un autrenotebook occupe ton quota GPU. Mets l'accélérateur sur **None**.**La question.** Mes mesures sur 24 datasets montrent que le gain du fine-tuning dépend durapport entre la taille du contexte d'entraînement et celle du contexte d'inférence :23 datasets sur 24 améliorés quand ce rapport est proche de 1, et 4 sur 24 quand lecontexte de déploiement le dépasse d'un facteur 4 ou plus. Reste à savoir **ce que labibliothèque fait quand l'utilisateur ne précise rien.**Trois issues possibles, et elles n'ont pas la même portée :| Ce qu'on trouve | Ce que ça veut dire ||:--|:--|| un défaut **constant** (1024, 2048, …) indépendant des données | l'utilisateur par défaut est hors diagonale : c'est un défaut de réglage, avec des mesures à l'appui || un défaut qui prend **tout le jeu de données** | la bibliothèque est sur la diagonale par défaut ; le résultat ne concerne que ceux qui règlent ce paramètre à la main — beaucoup plus faible || le paramètre **n'est pas exposé** | le comportement est imposé ; reste à savoir lequel |La sortie de ce notebook est citable telle quelle dans un article ou un rapport de bug :elle affiche la version, les chemins de fichiers et les numéros de ligne.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0'],check=True)
print('tabpfn installé.')

## 1. Version et emplacement du paquet

In [ ]:

import importlib.metadata, pathlib, inspect, re, textwrap, json
import tabpfn

VERSION = importlib.metadata.version('tabpfn')
ROOT = pathlib.Path(tabpfn.__file__).parent
print('tabpfn        :', VERSION)
print('emplacement   :', ROOT)
print('sous-modules  :', sorted(p.name for p in ROOT.iterdir() if p.is_dir() and not p.name.startswith('_')))

def show(obj, nom):
    try:
        print(f'\n--- {nom} ---')
        print(inspect.signature(obj))
    except Exception as exc:
        print(f'\n--- {nom} : indisponible ({type(exc).__name__}: {exc}) ---')

def doc(obj, nom, n=2500):
    d = inspect.getdoc(obj) or '(aucune docstring)'
    print(f'\n===== docstring : {nom} =====')
    print(d[:n] + ('…' if len(d) > n else ''))

## 2. L'API de fine-tuningLes signatures et les valeurs par défaut. C'est ici que se trouve la réponse : la taille dusupport d'entraînement est-elle une constante, ou dépend-elle des données ?

In [ ]:

TARGETS = []
try:
    from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
    TARGETS.append((FinetunedTabPFNClassifier.__init__,'FinetunedTabPFNClassifier.__init__'))
    TARGETS.append((FinetunedTabPFNClassifier.fit,'FinetunedTabPFNClassifier.fit'))
except Exception as exc:
    print('FinetunedTabPFNClassifier introuvable :',type(exc).__name__,exc)
    FinetunedTabPFNClassifier=None
try:
    from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
    TARGETS.append((get_preprocessed_dataset_chunks,'get_preprocessed_dataset_chunks'))
except Exception as exc:
    print('data_util introuvable :',type(exc).__name__,exc)
    get_preprocessed_dataset_chunks=None

for obj,nom in TARGETS: show(obj,nom)

print('\n\n==================== VALEURS PAR DÉFAUT ====================')
for obj,nom in TARGETS:
    try: sig=inspect.signature(obj)
    except Exception: continue
    print(f'\n{nom}')
    for p in sig.parameters.values():
        if p.name in ('self','args','kwargs'): continue
        d = '(obligatoire)' if p.default is inspect._empty else repr(p.default)
        print(f'   {p.name:<34} = {d}')

if FinetunedTabPFNClassifier is not None:
    doc(FinetunedTabPFNClassifier,'FinetunedTabPFNClassifier')
if get_preprocessed_dataset_chunks is not None:
    doc(get_preprocessed_dataset_chunks,'get_preprocessed_dataset_chunks')

## 3. Balayage du code sourceTout ce qui ressemble à une taille de contexte, de support ou de lot, avec fichier etnuméro de ligne — pour pouvoir citer.

In [ ]:

PAT = re.compile(r'(max_data_size|max_samples|support|context_size|n_context|chunk_size'
                 r'|max_context|n_samples_|MAX_|LIMIT|subsample)')
def balayer(dossier,titre):
    d = ROOT/dossier if dossier else ROOT
    if not d.exists(): print(f'\n{titre} : dossier absent'); return
    print(f'\n=================== {titre} ===================')
    for p in sorted(d.rglob('*.py')):
        lignes=[]
        for i,line in enumerate(p.read_text(errors='replace').splitlines(),1):
            s=line.strip()
            if not s or s.startswith('#'): continue
            if PAT.search(s) and ('=' in s or 'def ' in s):
                lignes.append((i,s))
        if lignes:
            print(f'\n  {p.relative_to(ROOT)}')
            for i,s in lignes[:40]:
                print(f'     {i:>5} | {s[:118]}')
            if len(lignes)>40: print(f'     … {len(lignes)-40} autres lignes')

balayer('finetuning','MODULE FINETUNING')
balayer('constants.py' if (ROOT/'constants.py').exists() else '','CONSTANTES (racine)')

## 4. Le côté inférencePar symétrie : que fait `TabPFNClassifier` quand on lui donne plus de lignes que ses limitesde pré-entraînement ? C'est l'autre moitié du rapport.

In [ ]:

try:
    from tabpfn import TabPFNClassifier
    show(TabPFNClassifier.__init__,'TabPFNClassifier.__init__')
    sig=inspect.signature(TabPFNClassifier.__init__)
    print('\nvaleurs par défaut pertinentes :')
    for nom in ['n_estimators','fit_mode','memory_saving_mode','ignore_pretraining_limits',
                'inference_config','random_state','device']:
        if nom in sig.parameters:
            p=sig.parameters[nom]
            print(f'   {nom:<28} = {"(obligatoire)" if p.default is inspect._empty else repr(p.default)}')
    d=inspect.getdoc(TabPFNClassifier) or ''
    for cle in ['ignore_pretraining_limits','pretraining','context','subsample','10000','memory']:
        for para in d.split('\n\n'):
            if cle in para:
                print(f'\n--- docstring, mention de « {cle} » ---')
                print(textwrap.fill(para.strip()[:700],100)); break
except Exception as exc:
    print('TabPFNClassifier :',type(exc).__name__,exc)

## 5. VerdictClassement automatique dans l'une des trois issues. À lire avec la sortie des sectionsprécédentes sous les yeux : le verdict est une aide, pas une preuve.

In [ ]:

verdict=[]
def defaut_de(obj,noms):
    try: sig=inspect.signature(obj)
    except Exception: return None
    for n in noms:
        if n in sig.parameters:
            p=sig.parameters[n]
            return (n, None if p.default is inspect._empty else p.default)
    return None

CANDIDATS=['max_data_size','max_samples','support_size','context_size','max_context_size',
           'n_context','chunk_size']
trouve=[]
for obj,nom in TARGETS:
    d=defaut_de(obj,CANDIDATS)
    if d: trouve.append((nom,)+d)

print('paramètres de taille exposés :')
if not trouve:
    print('   AUCUN — le paramètre n est pas exposé dans ces signatures.')
    verdict.append("CAS 3 : la taille du support n'est pas un paramètre utilisateur de cette API. "
                   "Regarder la section 3 pour savoir ce que le code impose.")
for nom,par,val in trouve:
    print(f'   {nom}.{par} = {val!r}')
    if val is None:
        verdict.append(f"CAS 2 (faible) : {nom}.{par} vaut None par défaut — à vérifier en "
                       "section 3, mais cela signifie probablement « prendre tout le jeu de "
                       "données », donc la bibliothèque est sur la diagonale par défaut.")
    elif isinstance(val,(int,float)):
        verdict.append(f"CAS 1 (fort) : {nom}.{par} vaut {val} par défaut, une constante "
                       "indépendante des données. Un utilisateur qui fine-tune sans rien "
                       f"préciser puis prédit avec plus de {val} lignes de contexte est hors "
                       "diagonale sans l avoir choisi.")
    else:
        verdict.append(f"À examiner à la main : {nom}.{par} = {val!r}")

print('\n==================== VERDICT ====================')
for v in verdict or ["Rien de concluant automatiquement : lire les sections 2 et 3."]:
    print('\n* '+textwrap.fill(v,96).replace('\n','\n  '))
print('\nVersion inspectée : tabpfn',VERSION)
print('À citer avec le fichier et le numéro de ligne affichés en section 3.')

## Ce qu'il faut en faire**Si c'est le cas 1** — un défaut constant — c'est le résultat le plus utile de la semaine :il transforme une observation empirique en défaut de réglage reproductible, avec la mesurede son coût (−5,9 % de log-loss relative, 4 datasets sur 24 améliorés, au contexte completavec des épisodes de 102 lignes). Le correctif tient en une ligne : faire dépendre la tailledu support de fine-tuning du contexte d'inférence attendu, ou la tirer au hasard sur laplage de déploiement.**Si c'est le cas 2** — le défaut prend tout le jeu de données — la bibliothèque fait déjàce qu'il faut, comme den Breejen. Le résultat reste vrai mais ne concerne que ceux quifixent ce paramètre à la main, et il faut le dire ainsi dans l'article plutôt que de laisserentendre que le défaut est fautif.**Dans les deux cas**, garde la sortie complète : la version, les chemins et les numéros deligne sont ce qui rend l'affirmation vérifiable par un relecteur.